In [0]:
from datetime import datetime
from pyspark.sql.functions import lit, current_timestamp

In [0]:
dbutils.widgets.text("batch_id" , "1" , "Batch IdD (1,2 or 3)")

In [0]:
batch_id = dbutils.widgets.get("batch_id")

print(f"batch_id: {batch_id}")


In [0]:
## initilize the 

team_name = "team_lemma"
catalog     = f"charles_schwab_retailbrokerage_dev_{team_name}"
landing_volume = f"/Volumes/{catalog}/landing/pwg"
Batch_Folder = f"Batch{batch_id}"

landing_path = f"{landing_volume}/{Batch_Folder}/finwire"
Bronze_table = f"{catalog}.bronze.finwire"

In [0]:
## finwire has only one file present in the batch 

if batch_id !="1":
    dbutils.notebook.exit(F"Finewire file not present in the {batch_id}")

In [0]:
try:
    run_info_now = (
        spark.read.parquet(landing_path)
             .select("_run_id", "_batch")
             .limit(1)
             .first()
    )
    
    carried_run_id = run_info_now[0] if run_info_now[0] else "Unknown"
    carried_batch = run_info_now[1] if run_info_now[1] else batch_id
except Exception:
    carried_run_id = "Unknown"
    carried_batch = batch_id

run_id = carried_run_id
print(run_id)

## read from landing

In [0]:
files = dbutils.fs.ls(landing_path)
parquet_files = [f for f in files if f.name.startswith("part-")]

print(f"Path          : {landing_path}")
print(f"Parquet files : {len(parquet_files)}")

assert len(parquet_files) > 0, \
        f"No parquet files at {landing_path} — run raw_to_landing first"

df_landing = spark.read.parquet(landing_path)

landing_count = df_landing.count()
print(f"Landing rows read : {landing_count}")
print(f"Landing columns   : {df_landing.columns}")


In [0]:
df_bronze = df_landing\
               .drop("_landing_ts")\
               .withColumn("_ingest_ts" , current_timestamp())\
               .withColumn("_batch",     lit(batch_id))\
                .withColumn("_run_id",    lit(run_id))


print("bronze schema")

df_bronze.printSchema()

               
               

### write bronze delta

In [0]:
from pyspark.sql import Row

In [0]:
recon_results = []

source_count = df_bronze.count()

df_bronze.write.format("delta")\
               .mode("append")\
                .partitionBy("_batch")\
               .saveAsTable(Bronze_table)


bronze_count = (
        spark.read.table(Bronze_table)
        .filter(f"_batch = '{batch_id}' AND _run_id = '{run_id}'")
        .count()
    )


status = "MATCH" if source_count == bronze_count else "MISMATCH"
print(f"Source rows  : {source_count}")
print(f"Bronze rows  : {bronze_count}")
print(f"Status       : {status}")



          

In [0]:
%run ../../02_common_utils/operations

In [0]:
recon_results.append(Row(
        source_table    = "finwire",
        batch_id        = Batch_Folder,
        source_count    = source_count,
        target_count    = bronze_count,
        status          = status,
        columns_applied = "raw_line"
    ))

In [0]:
from pyspark.sql import Row


recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if row.status in ("Match" , "Not Match"):
        log_pipeline_recon(
            spark         = spark,
            run_id        = run_id,
            batch_id      = row.batch_id,
            domain        = "MARKET",
            table_name    = row.source_table,
            source_layer  = "landing",
            target_layer  = "bronze",
            source_count  = row.source_count,
            target_count  = row.target_count
        )

        log_audit_event(
            spark = spark,
            run_id = run_id,
            batch = row.batch_id,
            layer = "bronze",
            table_name = row.source_table,
            operation = "Append",
            rows_affected  = row.landing_count
        )

display(recon_df)